# Week 3

In [2]:
import pandas as pd
list_data = pd.read_csv("W2_list_data.csv")
list_data
sold_data = pd.read_csv("W2_sold_data.csv")
sold_data

/var/folders/gn/t0769g993173xwvtkqwcs0z80000gn/T/ipykernel_67195/2921105911.py:4: DtypeWarning: Columns (0,1,51,63,64,65,67,68) have mixed types. Specify dtype option on import or set low_memory=False.
  sold_data = pd.read_csv("W2_sold_data.csv")


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,OriginatingSystemName,OriginatingSystemSubName,BuyerAgencyCompensationType,BuyerAgencyCompensation,latfilled,lonfilled
0,Mlslistings,Mlslistings,"Carpet,Tile,Wood",True,False,499000.0,551985747,jwachter@cbnorcal.com,2024-01-26,240000.0,...,Other,94401,6472.0,NaN,CRMLS,CRMLS_MLSL,NaN,NaN,NaN,NaN
1,SanDiego,SanDiego,NaN,False,False,759900.0,522107581,mdarwich12@gmail.com,2024-01-05,815000.0,...,NaN,91950,NaN,NaN,CRMLS,CRMLS_SAND,NaN,NaN,NaN,NaN
2,SanDiego,SanDiego,NaN,False,False,739900.0,510919001,mdarwich12@gmail.com,2024-01-05,810000.0,...,NaN,91950,NaN,NaN,CRMLS,CRMLS_SAND,NaN,NaN,NaN,NaN
3,Mlslistings,Mlslistings,NaN,False,NaN,NaN,1079166779,davidmartz@compass.com,2024-01-30,858000.0,...,Palm Springs Unified,92262,NaN,13504.0,CRMLS,CRMLS_MLSL,NaN,NaN,NaN,NaN
4,Southland,Southland,NaN,False,False,1890500.0,1075037759,karen.klein@theagencyre.com,2024-01-29,1890500.0,...,Los Angeles Unified,91356,0.0,17873.0,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
414562,LakeCounty,LakeCounty,NaN,True,False,279000.0,1069708148,jessica.homesells@gmail.com,2026-04-01,215000.0,...,Konocti Unified,95424,0.0,10019.0,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN
414563,TheInlandGateway,TheInlandGateway,"Carpet,Tile,Wood",False,False,449000.0,1067010583,brian@cohen-realty.com,2026-04-13,415000.0,...,Rim of the World,92322,0.0,12096.0,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN
414564,SanDiego,TriCounties,Vinyl,True,False,1579000.0,1063196193,info@homecoin.com,2026-04-16,1355000.0,...,Sanger Unified,93619,0.0,859874.4,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN
414565,Mrmls,OrangeCounty,NaN,True,False,855000.0,1062106215,jefflachef@gmail.com,2026-04-16,820000.0,...,Capistrano Unified,92629,0.0,2733.0,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN


In [31]:
import pandas as pd

mortgage = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US",parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']
mortgage


,date,rate_30yr_fixed
0,1971-04-02,7.33
1,1971-04-09,7.31
2,1971-04-16,7.31
3,1971-04-23,7.31
4,1971-04-30,7.29
...,...,...
2880,2026-06-11,6.52
2881,2026-06-18,6.47
2882,2026-06-25,6.49
2883,2026-07-02,6.43


In [32]:
# Step 2 – Resample weekly rates to monthly averages
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = (
mortgage.groupby('year_month')['rate_30yr_fixed']
.mean()
.reset_index()
)

In [33]:
# Step 3 – Create a matching year_month key on the MLS datasets
# Sold dataset — key off CloseDate
sold_data['year_month'] = pd.to_datetime(sold_data['CloseDate']).dt.to_period('M')
# Listings dataset — key off ListingContractDate
list_data['year_month'] = pd.to_datetime(
list_data['ListingContractDate']
).dt.to_period('M')

In [34]:
# Step 4 – Merge
sold_with_rates = sold_data.merge(mortgage_monthly, on='year_month', how='left')
listings_with_rates = list_data.merge(mortgage_monthly, on='year_month', how='left')

In [35]:
# Step 5 – Validate the merge
# Check for any unmatched rows (rate should not be null)
print(sold_with_rates['rate_30yr_fixed'].isnull().sum())
print(listings_with_rates['rate_30yr_fixed'].isnull().sum())


0
0


In [37]:
print(sold_with_rates[
['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']].head())

    CloseDate year_month  ClosePrice  rate_30yr_fixed
0  2024-01-26    2024-01    240000.0           6.6425
1  2024-01-05    2024-01    815000.0           6.6425
2  2024-01-05    2024-01    810000.0           6.6425
3  2024-01-30    2024-01    858000.0           6.6425
4  2024-01-29    2024-01   1890500.0           6.6425


In [38]:
#save both list and sold dataset as new csv
sold_with_rates.to_csv("W3_sold_enriched.csv", index=False)
listings_with_rates.to_csv("W3_list_enriched.csv", index=False)